# P6 Count-Camp Merge Audit

This notebook implements **Phase P6** from the frozen pipeline.

## Goals

- compute centroid similarity between the original count communities
- identify merge-eligible pairs under a transparent audit rule
- draft an optional higher-order count camp structure for the main text
- preserve the original algorithmic count communities in the analysis outputs
- create a count-community to main-text camp crosswalk if the audit supports compression

## Required outputs

- `artifacts/tables/p6_count_community_similarity_pairs.csv`
- `artifacts/tables/p6_count_higher_order_merge_audit.csv`
- `artifacts/tables/p6_count_community_to_camp_crosswalk.csv`
- `artifacts/reports/p6_count_merge_report.md`

Additional support output generated here:

- `artifacts/runtime/p6_count_merge_manifest.json`

This notebook evaluates narrative compression only. It does not alter the underlying algorithmic community outputs created in `P3`.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd()
if not (ROOT / 'artifacts' / 'tables' / 'p3_count_dominant_mode_profiles.csv').exists():
    ROOT = ROOT.parent

TABLES = ROOT / 'artifacts' / 'tables'
REPORTS = ROOT / 'artifacts' / 'reports'
RUNTIME = ROOT / 'artifacts' / 'runtime'

for path in [TABLES, REPORTS, RUNTIME]:
    path.mkdir(parents=True, exist_ok=True)

PIPELINE_VERSION = 'A1_GSTP2026Feb_contract_v1'
RUN_ID = 'A1_GSTP2026Feb_contract_v1__20260320__P6__nogit'
TARGET_MAIN_TEXT_COUNT_CAMPS = 5
MIN_RECOMMENDED_PAIR_SIMILARITY = 0.95
MIN_RECOMMENDED_DISCRIMINATIVE_SIMILARITY = 0.95
MIN_RECOMMENDED_GAIN_PROXY = 0.0

manifest = {
    'notebook': 'P6_Count_Camp_Merge_Audit.ipynb',
    'pipeline_version': PIPELINE_VERSION,
    'run_id': RUN_ID,
    'inputs': {
        'count_profiles': str(TABLES / 'p3_count_dominant_mode_profiles.csv'),
        'count_membership': str(TABLES / 'p3_count_community_membership.csv'),
        'p3_candidate_map': str(TABLES / 'p3_count_higher_order_camp_map.csv'),
        'count_mode_audit': str(TABLES / 'p5_mode_distinction_count.csv'),
    },
    'merge_rules': {
        'target_main_text_count_camps': TARGET_MAIN_TEXT_COUNT_CAMPS,
        'min_pair_similarity': MIN_RECOMMENDED_PAIR_SIMILARITY,
        'min_discriminative_similarity': MIN_RECOMMENDED_DISCRIMINATIVE_SIMILARITY,
        'min_gain_proxy': MIN_RECOMMENDED_GAIN_PROXY,
    },
}
(RUNTIME / 'p6_count_merge_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

display(Markdown(f"**Run ID:** `{RUN_ID}`"))
display(Markdown('**Inputs:** `P3` count communities + `P5` count mode audit'))


## Load Inputs and Verify the Original Count Community State

The merge audit must start from the original count communities already frozen in `P3`. It cannot redefine or overwrite the algorithmic baseline.


In [ ]:
count_profiles = pd.read_csv(TABLES / 'p3_count_dominant_mode_profiles.csv')
count_membership = pd.read_csv(TABLES / 'p3_count_community_membership.csv')
p3_candidate_map = pd.read_csv(TABLES / 'p3_count_higher_order_camp_map.csv')
count_mode_audit = pd.read_csv(TABLES / 'p5_mode_distinction_count.csv')

profile_columns = [col for col in count_profiles.columns if col.startswith('profile__')]
discriminative_modes = count_mode_audit.loc[count_mode_audit['mode_class'] == 'discriminative', 'mode'].tolist()
discriminative_profile_columns = [f'profile__{mode}' for mode in discriminative_modes]

if count_profiles['community'].nunique() != len(count_profiles):
    raise ValueError('Count community profile file contains duplicate community rows.')
if count_membership['community'].nunique() != len(count_profiles):
    raise ValueError('Count membership and count profile files disagree on the number of communities.')
if len(count_membership['Country/Area'].unique()) != 94:
    raise ValueError('Count membership no longer covers the frozen retained country set.')
if len(profile_columns) != 16:
    raise ValueError('Count community profiles no longer contain all 16 mode columns.')
if len(discriminative_profile_columns) == 0:
    raise ValueError('Count mode audit returned no discriminative modes, so merge audit cannot proceed sensibly.')

input_check = pd.DataFrame({
    'object': ['count_profiles', 'count_membership', 'count_mode_audit'],
    'rows': [len(count_profiles), len(count_membership), len(count_mode_audit)],
    'key_count': [count_profiles['community'].nunique(), count_membership['community'].nunique(), len(discriminative_modes)],
})
display(input_check)


## Helper Functions

The functions below score all possible community pairs, identify merge-eligible pairs, and draft a main-text crosswalk only if the audit supports compression.


In [ ]:
def parse_top_modes(top3_modes_text):
    return [part.split(' (')[0] for part in str(top3_modes_text).split(' / ')]


def top_mode_name_from_profile(profile):
    ranked = pd.Series(profile, dtype=float).sort_values(ascending=False)
    first_mode = ranked.index[0]
    second_mode = ranked.index[1]
    first_share = float(ranked.iloc[0])
    second_share = float(ranked.iloc[1])
    first_status, first_tier = first_mode.split(' | ', 1)
    second_status, second_tier = second_mode.split(' | ', 1)

    if first_status == 'operating' and first_tier == '1-5':
        if first_share >= 0.75:
            return 'Draft: high-purity operating 1-5'
        if second_mode == 'operating | 5-50' and second_share >= 0.25:
            return 'Draft: operating 1-5 / 5-50 mixed'
        return 'Draft: broad operating 1-5'
    if first_status == second_status and second_share >= 0.20:
        return f'Draft: {first_status} {first_tier} / {second_tier} mixed'
    if second_status != first_status and second_share >= 0.15:
        return f'Draft: {first_status} / {second_status} mixed ({first_tier}-led)'
    if first_share >= 0.45:
        return f'Draft: {first_status} {first_tier} heavy'
    return f'Draft: {first_status} {first_tier} led'


def build_similarity_pairs(profiles, profile_cols, discrim_cols):
    rows = []
    indexed = profiles.set_index('community')

    for i, left in enumerate(indexed.index[:-1]):
        left_row = indexed.loc[left]
        left_vector = left_row[profile_cols].to_numpy(dtype=float)
        left_discrim = left_row[discrim_cols].to_numpy(dtype=float)
        left_top3 = parse_top_modes(left_row['top3_modes'])

        for right in indexed.index[i + 1:]:
            right_row = indexed.loc[right]
            right_vector = right_row[profile_cols].to_numpy(dtype=float)
            right_discrim = right_row[discrim_cols].to_numpy(dtype=float)
            right_top3 = parse_top_modes(right_row['top3_modes'])

            pair_similarity = float(cosine_similarity(left_vector.reshape(1, -1), right_vector.reshape(1, -1))[0, 0])
            discriminative_similarity = float(cosine_similarity(left_discrim.reshape(1, -1), right_discrim.reshape(1, -1))[0, 0])

            w_left = int(left_row['countries'])
            w_right = int(right_row['countries'])
            merged_profile = (left_vector * w_left + right_vector * w_right) / (w_left + w_right)
            other_ids = [cid for cid in indexed.index if cid not in [left, right]]
            other_vectors = indexed.loc[other_ids, profile_cols].to_numpy(dtype=float) if other_ids else np.empty((0, len(profile_cols)))
            merged_max_similarity_to_other_camps = float(cosine_similarity(merged_profile.reshape(1, -1), other_vectors).max()) if len(other_ids) else np.nan
            merge_gain_proxy = pair_similarity - merged_max_similarity_to_other_camps if pd.notna(merged_max_similarity_to_other_camps) else np.nan

            rows.append({
                'left_community': int(left),
                'right_community': int(right),
                'left_name_draft': left_row['community_name_draft'],
                'right_name_draft': right_row['community_name_draft'],
                'left_dominant_mode': left_row['dominant_mode'],
                'right_dominant_mode': right_row['dominant_mode'],
                'pair_similarity_all_modes': pair_similarity,
                'pair_similarity_discriminative_modes': discriminative_similarity,
                'same_dominant_mode': bool(left_row['dominant_mode'] == right_row['dominant_mode']),
                'same_dominant_status': bool(left_row['dominant_status'] == right_row['dominant_status']),
                'top3_overlap': int(len(set(left_top3) & set(right_top3))),
                'left_countries': w_left,
                'right_countries': w_right,
                'countries_merged': int(w_left + w_right),
                'size_gap': int(abs(w_left - w_right)),
                'merged_dominant_mode': pd.Series(merged_profile, index=[c.replace('profile__', '') for c in profile_cols]).sort_values(ascending=False).index[0],
                'merged_dominant_mode_share': float(pd.Series(merged_profile, index=[c.replace('profile__', '') for c in profile_cols]).sort_values(ascending=False).iloc[0]),
                'merged_top3_share': float(pd.Series(merged_profile, index=[c.replace('profile__', '') for c in profile_cols]).sort_values(ascending=False).head(3).sum()),
                'merged_max_similarity_to_other_camps': merged_max_similarity_to_other_camps,
                'merge_gain_proxy': merge_gain_proxy,
            })

    pairs = pd.DataFrame(rows).sort_values(
        ['pair_similarity_all_modes', 'pair_similarity_discriminative_modes', 'top3_overlap', 'countries_merged'],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)
    return pairs


def score_merge_candidates(pairs):
    audit_rows = []
    for row in pairs.itertuples(index=False):
        if (
            row.pair_similarity_all_modes >= MIN_RECOMMENDED_PAIR_SIMILARITY
            and row.pair_similarity_discriminative_modes >= MIN_RECOMMENDED_DISCRIMINATIVE_SIMILARITY
            and row.same_dominant_mode
            and row.merge_gain_proxy > MIN_RECOMMENDED_GAIN_PROXY
        ):
            preliminary_support = 'recommended_candidate'
        elif row.pair_similarity_all_modes >= 0.90 and row.same_dominant_status and row.top3_overlap >= 2:
            preliminary_support = 'plausible'
        else:
            preliminary_support = 'not_recommended'

        audit_rows.append({
            'left_community': row.left_community,
            'right_community': row.right_community,
            'pair_similarity_all_modes': row.pair_similarity_all_modes,
            'pair_similarity_discriminative_modes': row.pair_similarity_discriminative_modes,
            'same_dominant_mode': row.same_dominant_mode,
            'same_dominant_status': row.same_dominant_status,
            'top3_overlap': row.top3_overlap,
            'countries_merged': row.countries_merged,
            'merge_gain_proxy': row.merge_gain_proxy,
            'preliminary_support': preliminary_support,
        })

    audit = pd.DataFrame(audit_rows)
    sort_cols = ['pair_similarity_all_modes', 'pair_similarity_discriminative_modes', 'merge_gain_proxy', 'top3_overlap', 'countries_merged']
    audit = audit.sort_values(sort_cols, ascending=[False, False, False, False, False]).reset_index(drop=True)

    audit['support_level'] = audit['preliminary_support'].replace({'recommended_candidate': 'plausible'})
    candidate_mask = audit['preliminary_support'] == 'recommended_candidate'
    if candidate_mask.any():
        top_idx = audit.loc[candidate_mask, sort_cols].sort_values(sort_cols, ascending=[False, False, False, False, False]).index[0]
        audit.loc[top_idx, 'support_level'] = 'recommended'

    audit['rationale'] = audit['support_level'].map({
        'recommended': 'Highest-support merge candidate under the frozen centroid audit.',
        'plausible': 'Potentially compressible, but weaker than the top candidate.',
        'not_recommended': 'Do not merge in the main-text compression layer.',
    })

    support_order = {'recommended': 0, 'plausible': 1, 'not_recommended': 2}
    audit['support_sort'] = audit['support_level'].map(support_order)
    audit = audit.sort_values(['support_sort'] + sort_cols, ascending=[True, False, False, False, False, False]).drop(columns=['support_sort', 'preliminary_support']).reset_index(drop=True)
    return audit


def build_crosswalk(profiles, selected_pair):
    indexed = profiles.set_index('community')
    left, right = selected_pair
    groups = [[cid] for cid in indexed.index if cid not in [left, right]] + [[left, right]]
    group_rows = []

    for group in groups:
        weights = indexed.loc[group, 'countries'].to_numpy(dtype=float)
        merged = np.average(indexed.loc[group, profile_columns].to_numpy(dtype=float), axis=0, weights=weights)
        profile = pd.Series(merged, index=[col.replace('profile__', '') for col in profile_columns])
        group_rows.append({
            'group_members': group,
            'countries': int(weights.sum()),
            'camp_name_draft': top_mode_name_from_profile(profile),
            'dominant_mode': profile.sort_values(ascending=False).index[0],
            'dominant_mode_share': float(profile.sort_values(ascending=False).iloc[0]),
            'top3_modes': ' / '.join([f'{mode} ({share:.2f})' for mode, share in profile.sort_values(ascending=False).head(3).items()]),
        })

    group_rows = sorted(group_rows, key=lambda row: (-row['countries'], row['group_members']))
    crosswalk_rows = []
    for camp_id, row in enumerate(group_rows, start=1):
        for community in row['group_members']:
            original = indexed.loc[community]
            crosswalk_rows.append({
                'original_count_community': int(community),
                'original_community_name_draft': original['community_name_draft'],
                'original_dominant_mode': original['dominant_mode'],
                'original_countries': int(original['countries']),
                'main_text_count_camp': int(camp_id),
                'main_text_count_camp_name_draft': row['camp_name_draft'],
                'camp_group_members': ', '.join(str(member) for member in row['group_members']),
                'camp_dominant_mode': row['dominant_mode'],
                'camp_dominant_mode_share': row['dominant_mode_share'],
                'camp_top3_modes': row['top3_modes'],
                'merge_applied': bool(len(row['group_members']) > 1),
            })

    crosswalk = pd.DataFrame(crosswalk_rows).sort_values('original_count_community').reset_index(drop=True)
    return crosswalk


## Compute Pairwise Similarity and Evaluate Merge Scenarios

The audit compares all possible community pairs and then selects a main-text crosswalk only if a merge candidate is clearly stronger than the alternatives.


In [ ]:
similarity_pairs = build_similarity_pairs(count_profiles, profile_columns, discriminative_profile_columns)
merge_audit = score_merge_candidates(similarity_pairs)
recommended_pairs = merge_audit.loc[merge_audit['support_level'] == 'recommended'].copy()

if recommended_pairs.empty:
    raise ValueError('No recommended count-community merge candidate was found under the frozen audit rules.')

selected_pair = tuple(recommended_pairs[['left_community', 'right_community']].iloc[0].tolist())
community_to_camp_crosswalk = build_crosswalk(count_profiles, selected_pair)

if community_to_camp_crosswalk['main_text_count_camp'].nunique() != TARGET_MAIN_TEXT_COUNT_CAMPS:
    raise ValueError('Selected merge candidate did not produce the expected 5-camp main-text crosswalk.')
if community_to_camp_crosswalk['original_count_community'].nunique() != count_profiles['community'].nunique():
    raise ValueError('Crosswalk does not cover all original count communities.')
if count_membership['community'].nunique() != count_profiles['community'].nunique():
    raise ValueError('Original algorithmic community count drift detected before merge audit.')

similarity_pairs.to_csv(TABLES / 'p6_count_community_similarity_pairs.csv', index=False)
merge_audit.to_csv(TABLES / 'p6_count_higher_order_merge_audit.csv', index=False)
community_to_camp_crosswalk.to_csv(TABLES / 'p6_count_community_to_camp_crosswalk.csv', index=False)

display(Markdown('**Top similarity pairs**'))
display(similarity_pairs.head(10))
display(Markdown('**Merge audit**'))
display(merge_audit.head(10))
display(Markdown('**Selected 5-camp crosswalk**'))
display(community_to_camp_crosswalk)


## Write the P6 Report

This report documents the evidence chain for the optional count-camp compression layer. The original algorithmic communities remain the analysis baseline regardless of whether the main text uses a compressed crosswalk.


In [ ]:
selected_pair_row = similarity_pairs.loc[
    (similarity_pairs['left_community'] == selected_pair[0]) & (similarity_pairs['right_community'] == selected_pair[1])
].iloc[0]

report_lines = [
    '# P6 Count Merge Report',
    '',
    '## Run metadata',
    f'- run_id: `{RUN_ID}`',
    f'- pipeline_version: `{PIPELINE_VERSION}`',
    '- inputs: `P3` count community profiles and memberships plus `P5` count mode-audit results',
    '',
    '## Interpretation boundary',
    '- This batch evaluates narrative compression only.',
    '- The original algorithmic count communities remain the analysis baseline and are not overwritten.',
    '',
    '## Current baseline state',
    f'- Frozen `P3` count output currently contains `{count_profiles["community"].nunique()}` algorithmic communities.',
    f'- This audit evaluates whether those communities can be summarized into `{TARGET_MAIN_TEXT_COUNT_CAMPS}` higher-order camps for the main text.',
    '',
    '## Recommended merge candidate',
    f'- selected pair: community {selected_pair[0]} + community {selected_pair[1]}',
    f'- pair similarity (all modes): {selected_pair_row["pair_similarity_all_modes"]:.6f}',
    f'- pair similarity (discriminative modes only): {selected_pair_row["pair_similarity_discriminative_modes"]:.6f}',
    f'- merge gain proxy: {selected_pair_row["merge_gain_proxy"]:.6f}',
    f'- merged dominant mode: {selected_pair_row["merged_dominant_mode"]}',
    f'- merged dominant mode share: {selected_pair_row["merged_dominant_mode_share"]:.6f}',
    '',
    '## Why this pair is preferred',
    '- It is the strongest pair under the frozen centroid similarity audit.',
    '- It shares the same dominant mode and the same dominant status across both communities.',
    '- It preserves a clean five-camp narrative layer without altering the underlying six-community output.',
    '',
    '## Main-text 5-camp crosswalk',
]
for row in community_to_camp_crosswalk.itertuples(index=False):
    report_lines.append(
        f'- original community {row.original_count_community} -> main-text camp {row.main_text_count_camp}: {row.main_text_count_camp_name_draft}; merge_applied={bool(row.merge_applied)}; members={row.camp_group_members}'
    )

report_lines.extend([
    '',
    '## Output inventory',
    '- `artifacts/tables/p6_count_community_similarity_pairs.csv`',
    '- `artifacts/tables/p6_count_higher_order_merge_audit.csv`',
    '- `artifacts/tables/p6_count_community_to_camp_crosswalk.csv`',
    '- `artifacts/reports/p6_count_merge_report.md`',
    '- `artifacts/runtime/p6_count_merge_manifest.json`',
])

(REPORTS / 'p6_count_merge_report.md').write_text('\n'.join(report_lines), encoding='utf-8')
display(Markdown((REPORTS / 'p6_count_merge_report.md').read_text(encoding='utf-8')))
